# M22 · Récurrences vectorielles, rayon spectral et normes de matrices — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m22_recurrences/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m22_recurrences/03_notebook.ipynb`, créée par `python tools/start_chapter.py M22`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M22.1 à M22.10, M22.16 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M22.19 | Les états d'une récurrence linéaire avec entrée · `linear_system_states` | 🔨 | ★ | 20 |
|  | M22.20 | Les normes induites ‖·‖₁, ‖·‖∞ et ‖·‖₂ · `operator_norm` | 🔨 | ★★ | 30 |
|  | M22.21 | Les normes des produits cumulés, et leur majoration · `product_norms` | 🔨 | ★ | 20 |
|  | M22.22 | `np.linalg.eigvals` et `np.linalg.norm` : rayon spectral et normes de matrices tirées au hasard, et ρ ≤ ‖A‖ | 📦 | ★ | 15 |
| B | M22.23 | Prédire l'allure de ‖Aᵗ‖₂ pour une matrice non normale de rayon spectral 0,8, puis tracer | 🔮 | ★ | 15 |
|  | M22.24 | Des produits de matrices aléatoires : le taux de croissance (1/n) ln ‖Aₙ ⋯ A₁‖ | 🔬 | ★★ | 20 |
|  | M22.25 | Les spirales d'une récurrence 2 × 2 à valeurs propres complexes, selon leur module | 🎨 | ★ | 15 |
| C | M22.26 | Chasse à l'erreur : ρ(A) pris pour ‖A‖, un produit de jacobiennes dans le mauvais ordre, un seuil appliqué bloc par bloc | 🐛 | ★ | 20 |
|  | M22.27 | Un test de propriété : ‖AB‖ ≤ ‖A‖ ‖B‖ et ρ(A) ≤ ‖A‖ sur des matrices tirées au hasard | 🛠️ | ★ | 15 |
| D | M22.28 | Défi : une matrice 3 × 3 à coefficients dans [−3, 3], de rayon spectral 0,85, dont une puissance a une norme ‖·‖₂ supérieure à 100 | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M22")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : une population avec des arrivées, une réponse impulsionnelle, une rotation-homothétie, trois rayons spectraux, une croissance transitoire, trois normes induites, des équipements qui s'usent, des jacobiennes de Hénon, la sensibilité d'un capital, un vecteur fait de blocs, et un pont vers la convention ligne.
- Écrire les trois fonctions de `mymath/recurrence.py` (`linear_system_states`, `operator_norm`, `product_norms`) et les tester.
- Comparer rayon spectral et normes sur des matrices tirées au hasard, prédire et tracer une croissance transitoire, mesurer le taux de croissance d'un produit de matrices aléatoires, dessiner des spirales, corriger trois erreurs classiques, écrire un test de propriété, et relever un défi.

**Rappel express.** $\mathbf{x}_{t+1} = \mathbf{A}\mathbf{x}_t + \mathbf{B}\mathbf{u}_t$ (colonnes) : $\mathbf{x}_t = \mathbf{A}^t\mathbf{x}_0 + \sum_{k<t}\mathbf{A}^{t-1-k}\mathbf{B}\mathbf{u}_k$ ; $h_k = \mathbf{C}\mathbf{A}^k\mathbf{B}$ et, pour $\mathbf{x}_0 = \mathbf{0}$, $y_{t+1} = (\mathbf{h} * \mathbf{u})_t$. $z = r\,e^{i\theta}$, $z^t = r^te^{it\theta}$ ; $\begin{pmatrix} a & -b \\ b & a \end{pmatrix} = r\mathbf{R}_\theta$ ; une matrice réelle $2 \times 2$ à valeurs propres complexes a pour module $\sqrt{\det\mathbf{A}}$. $\rho(\mathbf{A}) = \max\lvert\lambda_i\rvert$ ; $\mathbf{A}^t \to \mathbf{0} \iff \rho(\mathbf{A}) < 1$ ; une matrice non normale ($\mathbf{A}^\top\mathbf{A} \ne \mathbf{A}\mathbf{A}^\top$) peut grandir d'abord. $\lVert\mathbf{A}\rVert_1$ : colonnes ; $\lVert\mathbf{A}\rVert_\infty$ : lignes ; $\lVert\mathbf{A}\rVert_2 = s_{\max}$ ; $\lVert\mathbf{A}\mathbf{B}\rVert \le \lVert\mathbf{A}\rVert\,\lVert\mathbf{B}\rVert$, $\rho \le \lVert\cdot\rVert$, $\lVert\mathbf{A}^t\rVert^{1/t} \to \rho$ ; la norme de Frobenius n'est pas induite. $\mathbf{P}_t = \mathbf{A}_t\cdots\mathbf{A}_1$, $\lVert\mathbf{P}_t\rVert \le \prod \lVert\mathbf{A}_k\rVert$ ; $(1 - \varepsilon)^t \approx e^{-t\varepsilon}$ ; le signe de $\mathbb{E}[\ln a]$ décide. $\partial\mathbf{x}_T/\partial\mathbf{x}_t = \mathbf{J}_f(\mathbf{x}_{T-1})\cdots\mathbf{J}_f(\mathbf{x}_t)$ ; $\mathbf{s}_{t+1} = \mathbf{J}_t\mathbf{s}_t + \mathbf{b}_t$. Blocs : $\lVert\mathbf{w}\rVert^2 = \sum\lVert\mathbf{w}_k\rVert^2$, et $\mathbf{w}\min(1, c/\lVert\mathbf{w}\rVert)$ garde la direction.

## Partie 0 · Vérifier tes exercices papier (✏️ M22.1 à M22.10, 🔗 M22.16)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une liste `[0.2, -1]` ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ;
- un nombre complexe $a + ib$ comme la liste `[a, b]` ;
- une fraction irréductible entre guillemets, `"3/7"`, quand la question le dit ;
- un angle exact en radians entre guillemets, `"pi/4"` ou `"3*pi/5"` ;
- une **forme** comme un tuple, `(7, 2)`, ou comme un texte, `"(7, 2)"` ;
- `True` ou `False` pour un vrai ou faux.

Les réponses pas encore remplies affichent ⏳.

**Ex M22.1 — Une population à trois classes d'âge, avec des arrivées : trois pas à la main, et la solution écrite avec les puissances**

In [ ]:
# Ex M22.1: the values you found on paper
answer_M22_1a = ...  # a) a list of 3 numbers
answer_M22_1b = ...  # b) a list of 3 numbers
answer_M22_1c = ...  # c) a list of 3 numbers
answer_M22_1d = ...  # d) a 3 × 2 matrix (list of rows)
answer_M22_1e = ...  # e) a list of 3 numbers
answer_M22_1f = ...  # f) a list of 3 numbers
answer_M22_1g = ...  # g) an integer
answer_M22_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M22_1a, answer_M22_1b, answer_M22_1c, answer_M22_1d, answer_M22_1e, answer_M22_1f, answer_M22_1g, answer_M22_1h]):
    wbm.check(f"M22.1{letter}", answer)

**Ex M22.2 — La réponse impulsionnelle de deux réservoirs en cascade, et la sortie comme convolution de l'entrée**

In [ ]:
# Ex M22.2: the values you found on paper
answer_M22_2a = ...  # a) a list of 5 numbers (4 decimals)
answer_M22_2b = ...  # b) a list of 5 numbers (4 decimals)
answer_M22_2c = ...  # c) a list of 2 numbers
answer_M22_2d = ...  # d) an integer
answer_M22_2e = ...  # e) an irreducible fraction, as text "p/q"
answer_M22_2f = ...  # f) a number (4 decimals)
answer_M22_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M22_2a, answer_M22_2b, answer_M22_2c, answer_M22_2d, answer_M22_2e, answer_M22_2f, answer_M22_2g]):
    wbm.check(f"M22.2{letter}", answer)

**Ex M22.3 — Une matrice de rotation et d'homothétie : valeurs propres complexes, forme exponentielle, puissances**

In [ ]:
# Ex M22.3: the values you found on paper
answer_M22_3a = ...  # a) a list [a, b] for a + ib (4 decimals)
answer_M22_3b = ...  # b) a number
answer_M22_3c = ...  # c) an exact angle in radians, as text, like "pi/4"
answer_M22_3d = ...  # d) a 2 × 2 matrix (4 decimals)
answer_M22_3e = ...  # e) a list [a, b] for a + ib (4 decimals)
answer_M22_3f = ...  # f) a number (5 decimals)
answer_M22_3g = ...  # g) an exact angle in [0, 2π[, as text
answer_M22_3h = ...  # h) True or False
answer_M22_3i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M22_3a, answer_M22_3b, answer_M22_3c, answer_M22_3d, answer_M22_3e, answer_M22_3f, answer_M22_3g, answer_M22_3h, answer_M22_3i]):
    wbm.check(f"M22.3{letter}", answer)

**Ex M22.4 — Le rayon spectral de trois matrices 2 × 2 non symétriques, et le comportement de leurs puissances**

In [ ]:
# Ex M22.4: the values you found on paper
answer_M22_4a = ...  # a) a number
answer_M22_4b = ...  # b) a number
answer_M22_4c = ...  # c) a number
answer_M22_4d = ...  # d) a 2 × 2 matrix
answer_M22_4e = ...  # e) an integer
answer_M22_4f = ...  # f) a 2 × 2 matrix (4 decimals)
answer_M22_4g = ...  # g) a list of 3 integers (codes 1, 2 or 3)
answer_M22_4h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M22_4a, answer_M22_4b, answer_M22_4c, answer_M22_4d, answer_M22_4e, answer_M22_4f, answer_M22_4g, answer_M22_4h]):
    wbm.check(f"M22.4{letter}", answer)

**Ex M22.5 — Une matrice triangulaire de rayon spectral < 1 dont les puissances grandissent d'abord : Aᵗ en formule**

In [ ]:
# Ex M22.5: the values you found on paper
answer_M22_5a = ...  # a) an irreducible fraction, as text "p/q"
answer_M22_5b = ...  # b) a list of 5 numbers (4 decimals)
answer_M22_5c = ...  # c) a list of integers, in increasing order
answer_M22_5d = ...  # d) a number (4 decimals)
answer_M22_5e = ...  # e) an integer
answer_M22_5f = ...  # f) a list of 2 numbers (4 decimals)
answer_M22_5g = ...  # g) a list of 3 numbers (4 decimals)
answer_M22_5h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M22_5a, answer_M22_5b, answer_M22_5c, answer_M22_5d, answer_M22_5e, answer_M22_5f, answer_M22_5g, answer_M22_5h]):
    wbm.check(f"M22.5{letter}", answer)

**Ex M22.6 — Les normes ‖·‖₁, ‖·‖∞ et ‖·‖₂ d'une matrice 2 × 2, et trois majorations de ‖A⁵x‖**

In [ ]:
# Ex M22.6: the values you found on paper
answer_M22_6a = ...  # a) a number
answer_M22_6b = ...  # b) a number
answer_M22_6c = ...  # c) a 2 × 2 matrix
answer_M22_6d = ...  # d) a number (4 decimals)
answer_M22_6e = ...  # e) a number
answer_M22_6f = ...  # f) a list of 2 integers
answer_M22_6g = ...  # g) a list of 3 numbers (2 decimals)
answer_M22_6h = ...  # h) a list of 3 numbers (2 decimals)
answer_M22_6i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_M22_6a, answer_M22_6b, answer_M22_6c, answer_M22_6d, answer_M22_6e, answer_M22_6f, answer_M22_6g, answer_M22_6h, answer_M22_6i]):
    wbm.check(f"M22.6{letter}", answer)

**Ex M22.7 — Trois équipements qui s'usent à des rythmes différents : une récurrence diagonale, ce qui reste, et la demi-vie**

In [ ]:
# Ex M22.7: the values you found on paper
answer_M22_7a = ...  # a) a list of 3 numbers (4 decimals)
answer_M22_7b = ...  # b) a list of 3 numbers (4 decimals)
answer_M22_7c = ...  # c) a list of 3 numbers (2 decimals)
answer_M22_7d = ...  # d) a list of 3 numbers (2 decimals)
answer_M22_7e = ...  # e) a list of 3 numbers (4 decimals)
answer_M22_7f = ...  # f) a list of 3 numbers
answer_M22_7g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M22_7a, answer_M22_7b, answer_M22_7c, answer_M22_7d, answer_M22_7e, answer_M22_7f, answer_M22_7g]):
    wbm.check(f"M22.7{letter}", answer)

**Ex M22.8 — Les jacobiennes d'une application du plan en trois points d'une trajectoire, et leur produit**

In [ ]:
# Ex M22.8: the values you found on paper
answer_M22_8a = ...  # a) a list of 2 numbers
answer_M22_8b = ...  # b) a 2 × 2 matrix
answer_M22_8c = ...  # c) a 2 × 2 matrix
answer_M22_8d = ...  # d) a 2 × 2 matrix
answer_M22_8e = ...  # e) a number
answer_M22_8f = ...  # f) a list of 4 numbers
answer_M22_8g = ...  # g) a list of 2 numbers (5 decimals)
answer_M22_8h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M22_8a, answer_M22_8b, answer_M22_8c, answer_M22_8d, answer_M22_8e, answer_M22_8f, answer_M22_8g, answer_M22_8h]):
    wbm.check(f"M22.8{letter}", answer)

**Ex M22.9 — Un capital placé à taux fixe avec des versements : la dérivée du capital final par rapport au taux, une somme sur les années**

In [ ]:
# Ex M22.9: the values you found on paper
answer_M22_9a = ...  # a) a list of 4 numbers (2 decimals)
answer_M22_9b = ...  # b) a number
answer_M22_9c = ...  # c) a list of 4 numbers (2 decimals)
answer_M22_9d = ...  # d) a list of 4 numbers (2 decimals)
answer_M22_9e = ...  # e) a number (2 decimals)
answer_M22_9f = ...  # f) a number (2 decimals)
answer_M22_9g = ...  # g) a number (2 decimals)

for letter, answer in zip("abcdefg", [answer_M22_9a, answer_M22_9b, answer_M22_9c, answer_M22_9d, answer_M22_9e, answer_M22_9f, answer_M22_9g]):
    wbm.check(f"M22.9{letter}", answer)

**Ex M22.10 — La norme globale de trois blocs, et deux façons de les ramener sous un seuil**

In [ ]:
# Ex M22.10: the values you found on paper
answer_M22_10a = ...  # a) a list of 3 numbers
answer_M22_10b = ...  # b) a number
answer_M22_10c = ...  # c) a list of 3 numbers (4 decimals)
answer_M22_10d = ...  # d) a list of 3 numbers (4 decimals)
answer_M22_10e = ...  # e) a list of 3 numbers
answer_M22_10f = ...  # f) a number (4 decimals)
answer_M22_10g = ...  # g) a number (4 decimals)
answer_M22_10h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_M22_10a, answer_M22_10b, answer_M22_10c, answer_M22_10d, answer_M22_10e, answer_M22_10f, answer_M22_10g, answer_M22_10h]):
    wbm.check(f"M22.10{letter}", answer)

**Ex M22.16 — Pont : la convention ligne x_{t+1} = x_t A + u_t B, ses formes, et la jacobienne Aᵀ**

In [ ]:
# Ex M22.16: the values you found on paper
answer_M22_16a = ...  # a) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M22_16b = ...  # b) a shape: a tuple (7, 2) or a text "(7, 2)"
answer_M22_16c = ...  # c) a 3 × 2 matrix (4 decimals)
answer_M22_16d = ...  # d) a 2 × 2 matrix
answer_M22_16e = ...  # e) a 2 × 2 matrix
answer_M22_16f = ...  # f) a 2 × 2 matrix (4 decimals)
answer_M22_16g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_M22_16a, answer_M22_16b, answer_M22_16c, answer_M22_16d, answer_M22_16e, answer_M22_16f, answer_M22_16g]):
    wbm.check(f"M22.16{letter}", answer)

## Partie A · Ta librairie : les états d'une récurrence, les normes induites et les normes des produits

Fiche §1, §3 à §6. Les trois fonctions de `mymath/recurrence.py`, leurs tests, et un usage de chacune ; puis `np.linalg.eigvals` et `np.linalg.norm` sur des matrices tirées au hasard.

In [ ]:
# Tools of the notebook (parts A to D)
import math
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, run_pytest_group, same, verdict

TEST_M22 = "tests/test_m22_recurrence.py"            # the tests of mymath/recurrence.py


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        if isinstance(value, tuple):
            return "(" + ", ".join(str(np.round(np.asarray(v, dtype=float), digits).tolist()) for v in value) + ")"
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())
    except (TypeError, ValueError):
        return repr(value)[:120]


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def through(ax, xs, ys, rtol=1e-9):
    """True when one line of ax passes by all the points (xs, ys), in this order."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        x, y = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
        if (x.shape == xs.shape and y.shape == ys.shape and np.allclose(x, xs, rtol=rtol, atol=1e-12)
                and np.allclose(y, ys, rtol=rtol, atol=0)):
            return True
    return False


def horizontal(ax, level):
    """True when ax has a horizontal line at the height level (ax.axhline, or a line of constant height)."""
    for line in ax.lines:
        y = np.asarray(line.get_ydata(), dtype=float)
        if y.size >= 2 and np.allclose(y, level, rtol=1e-9, atol=1e-12):
            return True
    return False

### Ex M22.19 — Les états d'une récurrence linéaire avec entrée · `linear_system_states` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les états d'une récurrence linéaire avec entrée, puis s'en servir pour une population qui rejoint son équilibre et pour la sortie de deux bassins, une convolution.  
**Prérequis :** fiche §1 ; Ex M22.1 et M22.2 · **mymath :** `recurrence.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M03). Ouvre ta copie `mon_travail/mymath/recurrence.py`, lis la docstring du module puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. ⚠️ Le squelette n'importe rien : ajoute `import numpy as np` en tête du fichier (et `import numbers` si tu t'en sers). NumPy est permis, mais, de `np.linalg`, seulement `eigvalsh` (dans `operator_norm`) et `norm` (dans `product_norms`), et aucune fonction de SciPy : les tests comparent tes résultats à ceux de `np.linalg.norm`, `matrix_norm`, `svdvals` et `matrix_power`, et un test lit ton module. Les nombres se rendent en `float` de Python, les vecteurs et les matrices en tableaux NumPy de flottants ; ne modifie jamais les tableaux reçus.

Écris `linear_system_states(A, B, x0, inputs)` (fiche §1), en suivant sa docstring : les états $\mathbf{x}_0, \ldots, \mathbf{x}_T$ de $\mathbf{x}_{t+1} = \mathbf{A}\mathbf{x}_t + \mathbf{B}\mathbf{u}_t$, rangés en lignes dans un tableau de forme $(T + 1, n)$ (la ligne $t$ est $\mathbf{x}_t$), avec une multiplication par $\mathbf{A}$ par pas, sans jamais calculer $\mathbf{A}^t$. Les entrées $\mathbf{u}_0, \ldots, \mathbf{u}_{T-1}$ sont les lignes de `inputs`, de forme $(T, m)$ ; $T = 0$ s'écrit `np.zeros((0, m))`. Un tableau de flottants ; `ValueError` pour une matrice $\mathbf{A}$ qui n'est pas carrée, un $\mathbf{B}$ ou des entrées qui ne sont pas à deux dimensions, ou des formes de $\mathbf{B}$, de `x0` et des entrées qui ne vont pas avec $\mathbf{A}$.

**L'usage.** La cellule fournie donne la population de M22.1 (`A_POP_19`, `B_POP_19`, `X0_POP_19`, et ses trois entrées `U_POP_19`, une par ligne) et les deux bassins de M22.2 (`A_BASIN_19`, `B_BASIN_19`, `C_BASIN_19`). Avec ta fonction :
- a) $\mathbf{x}_3$ pour la population de M22.1, à comparer à ta réponse c) de M22.1 (une liste de 3 nombres) ;
- b) la même population, partie de $\mathbf{x}_0 = (40, 20, 30)$, avec les mêmes lâchers $\mathbf{u}_t = (20, 10)$ chaque année (`U_CONST_19`) pendant $T = 60$ ans : la population totale (la somme des trois classes) au printemps 60 (2 décimales dans ta copie) ;
- c) l'équilibre $\mathbf{x}^\star$, solution de $(\mathbf{I} - \mathbf{A})\mathbf{x}^\star = \mathbf{B}\mathbf{u}$ (`np.linalg.solve`), et sa population totale : une liste de 4 nombres, les trois coordonnées de $\mathbf{x}^\star$ puis leur somme ;
- d) vrai ou faux : les trois coordonnées de $\mathbf{x}_{60}$ sont à moins de 0,01 de celles de $\mathbf{x}^\star$ ;
- `h_19` : pour les bassins de M22.2, la réponse impulsionnelle $h_0, \ldots, h_6$, obtenue en simulant une impulsion ($u_0 = 1$, puis six pluies nulles, depuis $\mathbf{x}_0 = \mathbf{0}$) avec ta fonction, et en lisant $y_{t+1} = h_t$, avec $y_t = \mathbf{C}\mathbf{x}_t$ (7 nombres) ;
- e) les rejets $y_1, \ldots, y_7$ pour les pluies $(10, 0, 20, 5, 0, 0, 0)$ (`RAIN_19`), calculés à partir des états ($y_t = \mathbf{C}\mathbf{x}_t$) : vrai ou faux, ils sont égaux, à $10^{-12}$ près, aux sept premières valeurs de `np.convolve(h_19, RAIN_19)` ;
- f) $y_6$ et $y_7$ (une liste de 2 nombres ; 4 décimales dans ta copie).

La vérification compare aussi `h_19` à $\mathbf{C}\mathbf{A}^k\mathbf{B}$, calculé par NumPy. Ce qu'il faut observer : partie d'un autre état, la population oublie son départ et rejoint l'équilibre, au rythme de $\rho(\mathbf{A}) \approx 0{,}84$ par an ; et la sortie d'un système linéaire parti de $\mathbf{0}$ est la convolution de son entrée par sa réponse impulsionnelle, décalée d'un pas. Dans ta copie : pourquoi l'écart $\mathbf{x}_t - \mathbf{x}^\star$ est-il une réponse libre ? Pourquoi $h_0 = 0$ ?

In [ ]:
A_POP_19 = np.array([[0.0, 1.2, 0.8], [0.4, 0.0, 0.0], [0.0, 0.6, 0.0]])    # the population of M22.1
B_POP_19 = np.array([[1.0, 0.0], [0.0, 0.0], [0.0, 1.0]])
X0_POP_19 = np.array([40.0, 20.0, 30.0])
U_POP_19 = np.array([[20.0, 10.0], [0.0, 10.0], [30.0, 0.0]])          # u_0, u_1, u_2, one per row
U_CONST_19 = np.array([20.0, 10.0])                                     # the same releases every year (b to d)
A_BASIN_19 = np.array([[0.4, 0.0], [0.4, 0.3]])                         # the two basins of M22.2
B_BASIN_19 = np.array([[1.0], [0.0]])
C_BASIN_19 = np.array([0.0, 0.6])                                       # y_t = C x_t, the release to the river
RAIN_19 = np.array([10.0, 0.0, 20.0, 5.0, 0.0, 0.0, 0.0])               # the rains of the days 0 to 6 (mm)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M22")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M22, "test_linear_system_states_ or test_module_uses_")

**L'usage** : une population qui rejoint son équilibre, et deux bassins dont la sortie est une convolution.

In [ ]:
answer_M22_19a = answer_M22_19b = answer_M22_19c = answer_M22_19d = answer_M22_19e = answer_M22_19f = ...
h_19 = ...
with wbm.attempt("M22.19"):          # if you call linear_system_states here: ⏳ while it is not written
    answer_M22_19a = ...  # a) a list of 3 numbers, as computed: x_3 of the population of M22.1
    answer_M22_19b = ...  # b) a number, as computed (not rounded): the total population at spring 60, u_t = U_CONST_19
    answer_M22_19c = ...  # c) a list of 4 numbers: x* (np.linalg.solve), then its total population
    answer_M22_19d = ...  # d) True or False: the 3 coordinates of x_60 are within 0.01 of those of x*
    h_19 = ...            # the impulse response h_0, ..., h_6 of the basins, read on y_(t+1) = h_t (7 numbers)
    answer_M22_19e = ...  # e) True or False: y_1, ..., y_7 (from the states) equal np.convolve(h_19, RAIN_19)[:7] to 1e-12
    answer_M22_19f = ...  # f) a list of 2 numbers, as computed: y_6 and y_7

In [ ]:
with checked("M22.19"):
    if filled(h_19):
        same("M22.19", "h_19, la réponse impulsionnelle h_0 … h_6 des bassins", h_19,
             [float(C_BASIN_19 @ np.linalg.matrix_power(A_BASIN_19, k) @ B_BASIN_19[:, 0]) for k in range(7)])
    else:
        print("⏳ Ex M22.19 : la comparaison attend ton tableau h_19.")
for letter, answer in zip("abcdef", [answer_M22_19a, answer_M22_19b, answer_M22_19c, answer_M22_19d, answer_M22_19e,
                                     answer_M22_19f]):
    wbm.check(f"M22.19{letter}", answer, computed=True)

### Ex M22.20 — Les normes induites ‖·‖₁, ‖·‖∞ et ‖·‖₂ · `operator_norm` 🔨 ★★ ⏱️ 30 min
**Objectif :** programmer les trois normes induites qui se calculent, puis voir la formule de Gelfand, la norme de Frobenius et une croissance transitoire.  
**Prérequis :** fiche §4 ; M12d ; Ex M22.5 et M22.6 · **mymath :** `recurrence.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M22.19 : écris la fonction dans ta copie `mon_travail/mymath/recurrence.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `operator_norm(A, ord=2)` (fiche §4), en suivant sa docstring : la norme de $\mathbf{A}$ induite par la norme de vecteurs `ord`, la plus grande somme des valeurs absolues d'une **colonne** (`ord=1`), d'une **ligne** (`ord=np.inf`), ou $s_{\max}$, la racine de la plus grande valeur propre de $\mathbf{A}^\top\mathbf{A}$ (`ord=2`, avec `np.linalg.eigvalsh`, la valeur ramenée à 0 si l'arrondi la rend négative). Un `float` de Python ; $\mathbf{A}$ peut être rectangulaire ; `ValueError` pour un $\mathbf{A}$ qui n'est pas à deux dimensions ou qui est vide, ou un `ord` qui n'est pas 1, 2 ou `np.inf` (un booléen est refusé, et `"fro"` aussi : la norme de Frobenius n'est pas induite).

**L'usage.** La cellule fournie donne la matrice de M22.6 (`A6_20`) et la matrice $\mathbf{A}$ de M22.5 (`A5_20`). Avec ta fonction (et des produits successifs pour les puissances) :
- a) les trois normes de `A6_20`, dans l'ordre 1, 2, $\infty$ (une liste de 3 nombres) ;
- b) pour cette matrice, $\lVert\mathbf{A}^t\rVert_2^{1/t}$ pour $t = 1$, 10 et 50 (une liste de 3 nombres ; 4 décimales dans ta copie), à comparer à $\rho(\mathbf{A})$ (M22.6 e) ;
- c) `operator_norm(np.eye(5), ord)` pour `ord` = 1, 2 et `np.inf`, puis `np.linalg.norm(np.eye(5))` (une liste de 4 nombres) ;
- d) pour `A5_20`, le pas $t$ (de 1 à 10) où $\lVert\mathbf{A}^t\rVert_2$ est le plus grand (un entier) ;
- e) vrai ou faux : pour cette matrice, $\lVert\mathbf{A}^t\rVert_2 \ge (\mathbf{A}^t)_{12}$ pour $t = 1, \ldots, 30$ ($(\mathbf{A}^t)_{12}$, le coefficient en haut à droite, s'écrit `[0, 1]` en Python).

Ce qu'il faut observer : $\lVert\mathbf{A}^t\rVert^{1/t}$ descend lentement vers $\rho(\mathbf{A})$ (la formule de Gelfand) ; la norme de Frobenius de $\mathbf{I}_5$ n'est pas 1 : elle n'est pas induite ; et le maximum de la norme n'est pas toujours au même pas que celui d'un coefficient (d) contre M22.5 c). Dans ta copie : pourquoi une norme induite est-elle au moins égale à chaque coefficient en valeur absolue ?

In [ ]:
A6_20 = np.array([[1.0, 2.0], [-1.0, 2.0]])          # the matrix of M22.6
A5_20 = np.array([[2 / 3, 3.0], [0.0, 2 / 3]])        # the matrix A of M22.5

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M22")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M22, "test_operator_norm_")

**L'usage** : la formule de Gelfand, la norme de Frobenius, et une croissance transitoire.

In [ ]:
answer_M22_20a = answer_M22_20b = answer_M22_20c = answer_M22_20d = answer_M22_20e = ...
with wbm.attempt("M22.20"):          # if you call operator_norm here: ⏳ while it is not written
    answer_M22_20a = ...  # a) a list of 3 numbers, as computed: the norms 1, 2, inf of A6_20
    answer_M22_20b = ...  # b) a list of 3 numbers, as computed: ‖A6_20^t‖₂^(1/t) for t = 1, 10, 50
    answer_M22_20c = ...  # c) a list of 4 numbers: operator_norm(np.eye(5), ord) for 1, 2, inf, then np.linalg.norm(np.eye(5))
    answer_M22_20d = ...  # d) an integer: the step t (1 to 10) where ‖A5_20^t‖₂ is largest
    answer_M22_20e = ...  # e) True or False: ‖A5_20^t‖₂ >= (A5_20^t)[0, 1] for t = 1, ..., 30

In [ ]:
for letter, answer in zip("abcde", [answer_M22_20a, answer_M22_20b, answer_M22_20c, answer_M22_20d,
                                       answer_M22_20e]):
    wbm.check(f"M22.20{letter}", answer, computed=True)

### Ex M22.21 — Les normes des produits cumulés, et leur majoration · `product_norms` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les normes des produits cumulés d'une suite de matrices et leurs majorations, puis les suivre le long d'une trajectoire de Hénon.  
**Prérequis :** fiche §5 et §6 ; Ex M22.7 et M22.8 · **mymath :** `recurrence.py` · **Parcours :** C, E

> Même mode d'emploi qu'en M22.19 : écris la fonction dans ta copie `mon_travail/mymath/recurrence.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris `product_norms(matrices, ord=2)` (fiche §5), en suivant sa docstring : pour les matrices $\mathbf{A}_1, \ldots, \mathbf{A}_T$ données dans cet ordre (comme dans $\mathbf{x}_t = \mathbf{A}_t\mathbf{x}_{t-1}$), les normes des produits cumulés $\mathbf{P}_t = \mathbf{A}_t\cdots\mathbf{A}_1$ (le nouveau facteur **à gauche**) et leurs majorations $\lVert\mathbf{A}_t\rVert\cdots\lVert\mathbf{A}_1\rVert$, avec `np.linalg.norm` (permise ici). Le couple `(norms, bounds)`, deux tableaux de $T$ flottants ; `ValueError` sans matrice, pour une matrice qui n'est pas carrée ou des formes différentes, ou un `ord` qui n'est pas 1, 2 ou `np.inf`.

**L'usage.** La cellule fournie donne l'application de Hénon de M22.8 (`henon_21`) et sa jacobienne (`henon_jacobian_21`), et la matrice $\operatorname{diag}(0{,}985 ;\ 0{,}93 ;\ 0{,}65)$ des équipements de M22.7 (`WEAR_21`). Avec ta fonction :
- `states_21` : la trajectoire de Hénon partie de $(0, 0)$, sur 30 pas, $\mathbf{x}_0, \ldots, \mathbf{x}_{30}$ (31 lignes) ; `products_21` : ce que rend `product_norms` pour les jacobiennes $\mathbf{J}_f(\mathbf{x}_0), \ldots, \mathbf{J}_f(\mathbf{x}_{29})$, données dans cet ordre ;
- a) la troisième norme rendue, $\lVert\mathbf{P}_3\rVert_2$, qui est la norme de ta réponse d) de M22.8 (3 décimales dans ta copie) ;
- b) le rapport de la majoration à la norme pour $t = 30$, arrondi à l'entier ;
- c) le taux $\frac{1}{30}\ln\lVert\mathbf{P}_{30}\rVert_2$ (2 décimales dans ta copie) ;
- d) pour douze matrices `WEAR_21` : vrai ou faux, les normes et les majorations sont égales à $10^{-12}$ près (en valeur relative, `np.allclose(..., rtol=1e-12, atol=0)`) ;
- e) `draw_products_21(ax, norms, bounds)`, qui trace dans l'axe reçu (sans en ouvrir d'autre), en échelle logarithmique, les normes et les majorations contre $t = 1, \ldots, T$, chacune avec des points, avec une légende et un titre.

La vérification appelle `draw_products_21` sur les normes et les majorations de a) et regarde le dessin. Ce qu'il faut observer : la majoration par les normes grossit beaucoup plus vite que le produit quand les jacobiennes tournent les vecteurs dans des directions différentes (un facteur de l'ordre de mille après 30 pas) ; elle est exacte pour des matrices diagonales égales. Dans ta copie : pourquoi l'égalité de d), et pourquoi pas pour Hénon ?

In [ ]:
def henon_21(x):
    """The map of Hénon of M22.8: f(x, y) = (1 − 1.4 x² + y, 0.3 x)."""
    return np.array([1.0 - 1.4 * x[0] ** 2 + x[1], 0.3 * x[0]])


def henon_jacobian_21(x):
    """Its jacobian at the point x, one row per output coordinate (M17)."""
    return np.array([[-2.8 * x[0], 1.0], [0.3, 0.0]])


WEAR_21 = np.diag([0.985, 0.93, 0.65])            # the three pieces of equipment of M22.7

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M22")  # reload your saved file

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_M22, "test_product_norms_")

**L'usage** : les jacobiennes d'une trajectoire de Hénon, et des facteurs diagonaux.

In [ ]:
def draw_products_21(ax, norms, bounds):
    """In ax: the norms ‖P_t‖₂ and their bounds against t = 1, ..., len(norms), each with points, the y scale
    logarithmic; a legend and a title. No other axes."""
    raise NotImplementedError("draw_products_21 n'est pas encore écrite (04_indices.md si tu bloques)")


states_21 = products_21 = answer_M22_21a = answer_M22_21b = answer_M22_21c = answer_M22_21d = ...
with wbm.attempt("M22.21"):          # if you call product_norms here: ⏳ while it is not written
    states_21 = ...       # the 31 states x_0, ..., x_30 of the Hénon map from (0, 0), one per row
    products_21 = ...     # product_norms of the 30 jacobians J(x_0), ..., J(x_29), in this order: (norms, bounds)
    answer_M22_21a = ...  # a) a number, as computed: ‖P_3‖₂, the third norm
    answer_M22_21b = ...  # b) an integer: the ratio bound / norm at t = 30, rounded to the integer
    answer_M22_21c = ...  # c) a number, as computed: (1/30) ln ‖P_30‖₂
    answer_M22_21d = ...  # d) True or False: for twelve matrices WEAR_21, norms and bounds equal to 1e-12 (relative)

In [ ]:
for letter, answer in zip("abcd", [answer_M22_21a, answer_M22_21b, answer_M22_21c, answer_M22_21d]):
    wbm.check(f"M22.21{letter}", answer, computed=True)

with checked("M22.21"):
    results_21 = []
    report_21 = reporter(results_21)
    path_c21 = [np.zeros(2)]
    for _ in range(30):
        path_c21.append(np.array([1.0 - 1.4 * path_c21[-1][0] ** 2 + path_c21[-1][1], 0.3 * path_c21[-1][0]]))
    P_c21, bound_c21, norms_c21, bounds_c21 = np.eye(2), 1.0, [], []
    for x_c21 in path_c21[:30]:
        J_c21 = np.array([[-2.8 * x_c21[0], 1.0], [0.3, 0.0]])
        P_c21, bound_c21 = J_c21 @ P_c21, bound_c21 * float(np.linalg.svdvals(J_c21)[0])
        norms_c21.append(float(np.linalg.svdvals(P_c21)[0]))
        bounds_c21.append(bound_c21)
    steps_c21 = np.arange(1, 31)
    fig_21, ax_21 = plt.subplots(figsize=(7, 4.2))
    try:
        opened_21 = drawn(draw_products_21, ax_21, np.array(norms_c21), np.array(bounds_c21))
        report_21(not opened_21, "draw_products_21 dessine dans l'axe reçu",
                  "draw_products_21 doit dessiner dans l'axe reçu, sans ouvrir de figure")
        report_21(through(ax_21, steps_c21, norms_c21) and through(ax_21, steps_c21, bounds_c21),
                  "les normes et les majorations sont tracées contre t = 1, …, 30",
                  "il manque une courbe : les normes, puis les majorations, contre t = 1, …, 30 (une valeur par t)")
        report_21(ax_21.get_yscale() == "log", "l'échelle des ordonnées est logarithmique",
                  "mets l'échelle des ordonnées en logarithmique (ax.set_yscale(\"log\"))")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_21)
        raise
    plt.show()
    verdict("M22.21", all(results_21), "les normes des produits et leurs majorations sont dessinées.", "")

### Ex M22.22 — `np.linalg.eigvals` et `np.linalg.norm` : rayon spectral et normes de matrices tirées au hasard, et ρ ≤ ‖A‖ 📦 ★ ⏱️ 15 min
**Objectif :** comparer, avec NumPy, le rayon spectral et les normes de matrices tirées au hasard, et voir le piège de `np.linalg.norm` sans `ord`.  
**Prérequis :** fiche §3 et §4 · **Parcours :** C

La cellule fournie tire 1 000 matrices $4 \times 4$ de coefficients gaussiens (`A_22`, graine 5722 : `A_22[k]` est la $k$-ième). Calcule, pour chacune, son rayon spectral (`np.abs(np.linalg.eigvals(A_22[k])).max()`) et ses normes `np.linalg.norm(A_22[k], 1)`, `np.linalg.norm(A_22[k], 2)`, `np.linalg.norm(A_22[k], np.inf)` et `np.linalg.norm(A_22[k], "fro")`. Puis :
- a) la proportion des matrices pour lesquelles $\rho$ est au plus égal à la plus petite des trois normes induites ;
- b) la moyenne de $\frac{\rho}{\lVert\cdot\rVert_2}$ (2 décimales dans ta copie) ;
- c) la proportion des matrices pour lesquelles $\lVert\cdot\rVert_1 < \lVert\cdot\rVert_2$ (3 décimales dans ta copie) ;
- d) pour les parties symétriques $\mathbf{S}_k = \frac12(\mathbf{A}_k + \mathbf{A}_k^\top)$ : vrai ou faux, $\lvert\rho(\mathbf{S}_k) - \lVert\mathbf{S}_k\rVert_2\rvert < 10^{-12}$ pour les 1 000 ;
- e) pour `A_22[0]`, deux vrai ou faux, dans cet ordre (`np.isclose`) : `np.linalg.norm(A_22[0])` (sans `ord`) est égal à la norme de Frobenius, puis à $\lVert\cdot\rVert_2$ ;
- f) pour `A_22[0]`, $\lVert\mathbf{A}^t\rVert_2^{1/t}$ pour $t = 1$, 5, 25 et 100, puis $\rho$ (une liste de 5 nombres ; 3 décimales dans ta copie ; `np.linalg.matrix_power` est permise ici).

Ce qu'il faut observer : $\rho \le \lVert\cdot\rVert$ sans exception, avec un écart important pour une matrice quelconque (en moyenne, $\rho$ n'est que les deux tiers de $\lVert\cdot\rVert_2$) ; l'égalité pour les matrices symétriques (en norme 2) ; et le piège de `np.linalg.norm` sans `ord`. Dans ta copie : pourquoi l'égalité de d) ? Que dit f) de la formule de Gelfand ?

In [ ]:
rng_22 = np.random.default_rng(5722)
A_22 = rng_22.normal(size=(1000, 4, 4))                 # 1 000 matrices 4 × 4 of Gaussian coefficients

In [ ]:
answer_M22_22a = answer_M22_22b = answer_M22_22c = answer_M22_22d = answer_M22_22e = answer_M22_22f = ...
with wbm.attempt("M22.22"):
    answer_M22_22a = ...  # a) a number, as computed: the share of the matrices with ρ <= the smallest induced norm
    answer_M22_22b = ...  # b) a number, as computed: the mean of ρ / ‖·‖₂
    answer_M22_22c = ...  # c) a number, as computed: the share of the matrices with ‖·‖₁ < ‖·‖₂
    answer_M22_22d = ...  # d) True or False: |ρ(S_k) − ‖S_k‖₂| < 1e-12 for the 1 000 symmetric parts
    answer_M22_22e = ...  # e) a list of 2 True or False: np.linalg.norm(A_22[0]) equals the Frobenius norm, then ‖·‖₂
    answer_M22_22f = ...  # f) a list of 5 numbers, as computed: ‖A^t‖₂^(1/t) for t = 1, 5, 25, 100, then ρ (A = A_22[0])

In [ ]:
for letter, answer in zip("abcdef", [answer_M22_22a, answer_M22_22b, answer_M22_22c, answer_M22_22d,
                                      answer_M22_22e, answer_M22_22f]):
    wbm.check(f"M22.22{letter}", answer, computed=True)

## Partie B · Prédire, mesurer, dessiner

Fiche §2, §3 et §5. Une croissance transitoire, prédite puis tracée ; le taux de croissance d'un produit de matrices aléatoires, mesuré ; les spirales d'une récurrence à valeurs propres complexes, dessinées.

### Ex M22.23 — Prédire l'allure de ‖Aᵗ‖₂ pour une matrice non normale de rayon spectral 0,8, puis tracer 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir la croissance transitoire d'une matrice non normale de rayon spectral < 1, puis la mesurer et la tracer.  
**Prérequis :** fiche §3 et §4 ; Ex M22.20 · **Parcours :** C, E

$\mathbf{A} = \begin{pmatrix} -1{,}2 & 2 \\ -2 & 2{,}8 \end{pmatrix}$ (`A_23`). **Avant d'exécuter quoi que ce soit**, calcule à la main ses valeurs propres, $\mathbf{A}^\top\mathbf{A}$ et $\mathbf{A}\mathbf{A}^\top$, puis prédis :
- a) l'allure de $\lVert\mathbf{A}^t\rVert_2$ pour $t = 0$ à 40 : (A) elle décroît dès le premier pas, comme $0{,}8^t$ ; (B) elle monte pendant quelques pas, puis décroît vers 0 ; (C) elle monte sans fin ; (D) elle oscille sans décroître (une lettre) ;
- b) le pas $t$ de son maximum : (A) $t = 1$ ; (B) entre 3 et 6 ; (C) entre 15 et 25 (une lettre) ;
- c) vrai ou faux : $\lVert\mathbf{A}^{10}\rVert_2 > 1$ ;
- d) vrai ou faux : $\lVert\mathbf{A}^t\rVert_2 \ge 0{,}8^t$ pour tout $t$.

Puis, une fois tes prédictions écrites et vérifiées : calcule $\lVert\mathbf{A}^t\rVert_2$ pour $t = 0, \ldots, 40$ avec ta `operator_norm` (des produits successifs pour les puissances, $\mathbf{A}^0 = \mathbf{I}$), écris `draw_norms_23(ax, ts, norms)`, qui trace dans l'axe reçu (sans en ouvrir d'autre), en échelle logarithmique, les normes contre $t$ (des points reliés) et la courbe $0{,}8^t$ aux mêmes $t$, avec une légende ; et relève :
- e) le pas $t$ du maximum (un entier) ;
- f) la valeur du maximum (2 décimales dans ta copie) ;
- g) le premier pas $t$ où $\lVert\mathbf{A}^t\rVert_2 < 1$ (un entier).

La vérification de l'expérience ne s'exécute qu'une fois tes quatre prédictions écrites : elle compare tes normes à celles de NumPy, appelle `draw_norms_23` et regarde le dessin, en s'arrêtant au premier ❌. Ce qu'il faut observer : la valeur propre double 0,8 dit le long terme, pas les premiers pas ; la norme est multipliée par plus de 8 avant de décroître, et passe 20 pas au-dessus de 1.

In [ ]:
A_23 = np.array([[-1.2, 2.0], [-2.0, 2.8]])

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M22_23a = ...  # a) "A", "B", "C" or "D", written BEFORE the experiment: the shape of ‖A^t‖₂ for t = 0 to 40
prediction_M22_23b = ...  # b) "A", "B" or "C": the step of its maximum
prediction_M22_23c = ...  # c) True or False: ‖A^10‖₂ > 1
prediction_M22_23d = ...  # d) True or False: ‖A^t‖₂ >= 0.8^t for every t

In [ ]:
for letter, prediction in zip("abcd", [prediction_M22_23a, prediction_M22_23b, prediction_M22_23c,
                                        prediction_M22_23d]):
    wbm.check(f"M22.23{letter}", prediction)

**L'expérience.** Écris-la une fois tes quatre prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
def draw_norms_23(ax, ts, norms):
    """In ax: the norms ‖A^t‖₂ against t (points joined) and the curve 0.8^t at the same t, the y scale logarithmic;
    a legend. No other axes."""
    raise NotImplementedError("draw_norms_23 n'est pas encore écrite (04_indices.md si tu bloques)")


norms_23 = answer_M22_23e = answer_M22_23f = answer_M22_23g = ...
with wbm.attempt("M22.23"):          # your operator_norm: ⏳ while it is not written
    norms_23 = ...        # the 41 norms ‖A_23^t‖₂ for t = 0, ..., 40 (your operator_norm, successive products)
    answer_M22_23e = ...  # e) an integer: the step t of the largest norm
    answer_M22_23f = ...  # f) a number, as computed: the largest norm
    answer_M22_23g = ...  # g) an integer: the first step t where the norm is below 1

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M22_23a, prediction_M22_23b, prediction_M22_23c, prediction_M22_23d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    for letter, answer in zip("efg", [answer_M22_23e, answer_M22_23f, answer_M22_23g]):
        wbm.check(f"M22.23{letter}", answer, computed=True)

    with checked("M22.23"):
        results_23 = []
        report_23 = reporter(results_23)
        steps_c23 = np.arange(41)
        want_c23 = np.array([float(np.linalg.svdvals(np.linalg.matrix_power(A_23, t))[0]) for t in steps_c23])
        if not filled(norms_23):
            print("⏳ Ex M22.23 : la vérification attend tes normes norms_23.")
            raise StopCheck
        results_23.append(same("M22.23", "norms_23, les normes ‖Aᵗ‖₂ pour t = 0, …, 40", norms_23, want_c23))
        if not results_23[-1]:
            raise StopCheck
        fig_23, ax_23 = plt.subplots(figsize=(7, 4.2))
        try:
            opened_23 = drawn(draw_norms_23, ax_23, steps_c23, want_c23)
            report_23(not opened_23, "draw_norms_23 dessine dans l'axe reçu",
                      "draw_norms_23 doit dessiner dans l'axe reçu, sans ouvrir de figure")
            report_23(through(ax_23, steps_c23, want_c23), "les normes sont tracées contre t",
                      "il manque la courbe des normes contre t = 0, …, 40 (une valeur par t)")
            report_23(through(ax_23, steps_c23, 0.8 ** steps_c23), "la courbe 0,8ᵗ est tracée",
                      "il manque la courbe 0,8ᵗ, aux mêmes t")
            report_23(ax_23.get_yscale() == "log", "l'échelle des ordonnées est logarithmique",
                      "mets l'échelle des ordonnées en logarithmique (ax.set_yscale(\"log\"))")
        except StopCheck:
            plt.show()
            raise
        except BaseException:
            plt.close(fig_23)
            raise
        plt.show()
        verdict("M22.23", all(results_23), "les normes et 0,8ᵗ sont dessinées.", "")

**Après l'expérience**, dans ta copie : pourquoi la norme monte-t-elle d'abord, alors que les deux valeurs propres valent 0,8 ? Quel terme de la formule de Aᵗ d'un bloc de Jordan (fiche §3) le montre ? Que deviendrait l'expérience avec une matrice symétrique de mêmes valeurs propres ?

### Ex M22.24 — Des produits de matrices aléatoires : le taux de croissance (1/n) ln ‖Aₙ ⋯ A₁‖ 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer le taux de croissance d'un produit de matrices aléatoires, et le comparer au taux de la population moyenne et à la majoration par les normes.  
**Prérequis :** fiche §5 ; M02c, M06 ; Ex M22.21 · **Parcours :** C

Une plante vivace a deux stades, les graines dans le sol et les plantes en fleur : $\mathbf{x}_t = (\text{graines},\ \text{plantes})$. Une bonne année applique $\mathbf{G} = \begin{pmatrix} 0{,}2 & 3 \\ 0{,}6 & 0{,}4 \end{pmatrix}$ (`G_24`), une mauvaise $\mathbf{M} = \begin{pmatrix} 0{,}2 & 0{,}4 \\ 0{,}2 & 0{,}2 \end{pmatrix}$ (`M_24`), à pile ou face, indépendamment ; $n$ compte les années, c'est-à-dire les facteurs du produit $\mathbf{P}_n = \mathbf{A}_n\cdots\mathbf{A}_1$. La cellule fournie tire les années par `np.random.default_rng(5724).integers(0, 2, size=20000)` (`YEARS_24` ; 0 = bonne, 1 = mauvaise ; `YEARS_24[k - 1]` donne $\mathbf{A}_k$), et donne les graines de cinq autres suites, tirées de la même façon (`OTHER_SEEDS_24`, 5731 à 5735). Écris :
- `rates_24(years)` : les taux $\frac1n\sum_{k \le n}\ln\lVert\mathbf{x}_k\rVert_2$ pour $n = 1, \ldots$, `len(years)`, en renormalisant à chaque pas : partir de $\mathbf{x} = \frac{1}{\sqrt2}(1, 1)$, puis, chaque année, $\mathbf{x} \leftarrow \mathbf{A}_k\mathbf{x}$, ajouter $\ln\lVert\mathbf{x}\rVert_2$ à un cumul, et diviser $\mathbf{x}$ par sa norme ; c'est $\frac1n\ln\lVert\mathbf{P}_n\mathbf{x}_0\rVert_2$ ;
- `draw_rates_24(ax, curves, level_mean, level_bound)`, qui trace dans l'axe reçu (sans en ouvrir d'autre) chaque courbe de `curves` contre $n = 1, 2, \ldots$, l'abscisse logarithmique, et les droites horizontales `level_mean` et `level_bound` (`ax.axhline`), avec une légende.

Puis mesure :
- a) $\ln\rho(\mathbb{E}[\mathbf{A}])$, avec $\mathbb{E}[\mathbf{A}] = \frac12(\mathbf{G} + \mathbf{M})$ (3 décimales dans ta copie) : $\mathbb{E}[\mathbf{P}_n] = \mathbb{E}[\mathbf{A}]^n$, la population **moyenne**, grandit à ce taux ;
- b) avec ta `product_norms`, sur les 200 premières années, $\frac{1}{200}\ln\lVert\mathbf{P}_{200}\rVert_2$ (3 décimales dans ta copie) ;
- `curves_24` : les six courbes `rates_24`, celle de `YEARS_24` d'abord, puis celles des cinq autres suites, dans l'ordre de `OTHER_SEEDS_24` ;
- c) le taux pour $n = 20\,000$ sur `YEARS_24` (3 décimales dans ta copie) ;
- d) la majoration par les normes, $\mathbb{E}[\ln\lVert\mathbf{A}\rVert_2] = \frac12(\ln\lVert\mathbf{G}\rVert_2 + \ln\lVert\mathbf{M}\rVert_2)$ (3 décimales dans ta copie) ;
- e) le taux pour $n = 20\,000$ des cinq autres suites (une liste de 5 nombres ; 2 décimales dans ta copie) ;
- f) vrai ou faux : une trajectoire typique s'éteint, alors que la population moyenne grandit.

La vérification essaie `rates_24` sur des suites que tu ne vois pas, puis appelle `draw_rates_24` avec les six courbes et les droites a) et d), et regarde le dessin. Ce qu'il faut conclure : le taux converge vers le même nombre négatif pour toutes les suites (Furstenberg et Kesten, fiche §5) ; la population typique s'éteint, alors que la moyenne grandit, tirée par de rares suites très chanceuses ; et ni $\ln\rho(\mathbb{E}[\mathbf{A}])$, ni la majoration par les normes ne donnent ce taux : c'est la version matricielle de la plante du §5.

In [ ]:
G_24 = np.array([[0.2, 3.0], [0.6, 0.4]])            # a good year: (seeds, plants) multiplied by G
M_24 = np.array([[0.2, 0.4], [0.2, 0.2]])            # a bad year
YEARS_24 = np.random.default_rng(5724).integers(0, 2, size=20000)   # 0 = good, 1 = bad; YEARS_24[k - 1] gives A_k
OTHER_SEEDS_24 = [5731, 5732, 5733, 5734, 5735]      # five more sequences, drawn the same way

In [ ]:
def rates_24(years):
    """The running growth rates (1/n) Σ_(k<=n) ln ‖x_k‖₂ for n = 1, ..., len(years): from x = (1, 1)/√2, each year
    x <- A_k x (G_24 for a 0, M_24 for a 1), ln ‖x‖₂ added to the sum, then x <- x / ‖x‖₂. A 1-D array."""
    raise NotImplementedError("rates_24 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_rates_24(ax, curves, level_mean, level_bound):
    """In ax: each curve of `curves` against n = 1, ..., len(curve), the x scale logarithmic, and the horizontal
    lines level_mean and level_bound (ax.axhline); a legend. No other axes."""
    raise NotImplementedError("draw_rates_24 n'est pas encore écrite (04_indices.md si tu bloques)")


curves_24 = answer_M22_24a = answer_M22_24b = answer_M22_24c = answer_M22_24d = answer_M22_24e = answer_M22_24f = ...
with wbm.attempt("M22.24"):          # your functions and your product_norms: ⏳ while they are not written
    answer_M22_24a = ...  # a) a number, as computed: ln ρ(E[A]), E[A] = (G_24 + M_24) / 2
    answer_M22_24b = ...  # b) a number, as computed: (1/200) ln ‖P_200‖₂, your product_norms on the first 200 years
    curves_24 = ...       # the 6 arrays rates_24(...): YEARS_24 first, then the 5 sequences of OTHER_SEEDS_24
    answer_M22_24c = ...  # c) a number, as computed: the rate for n = 20 000 on YEARS_24
    answer_M22_24d = ...  # d) a number, as computed: (ln ‖G_24‖₂ + ln ‖M_24‖₂) / 2
    answer_M22_24e = ...  # e) a list of 5 numbers, as computed: the rate for n = 20 000 of the 5 other sequences
    answer_M22_24f = ...  # f) True or False: a typical trajectory dies out, while the mean population grows

In [ ]:
for letter, answer in zip("abcdef", [answer_M22_24a, answer_M22_24b, answer_M22_24c, answer_M22_24d,
                                      answer_M22_24e, answer_M22_24f]):
    wbm.check(f"M22.24{letter}", answer, computed=True)

with checked("M22.24"):
    results_24 = []
    report_24 = reporter(results_24)

    def rates_c24(years):
        """The running rates (1/n) Σ ln ‖x‖₂, renormalized at each step (the check)."""
        x, total, out = np.array([1.0, 1.0]) / math.sqrt(2), 0.0, []
        for n, year in enumerate(years, start=1):
            x = (G_24 if year == 0 else M_24) @ x
            size = math.hypot(x[0], x[1])
            total += math.log(size)
            x = x / size
            out.append(total / n)
        return np.array(out)

    rng_c24 = np.random.default_rng(5714)
    wrong_c24 = []
    for length_c24 in (1, 9, 300):
        years_c24 = rng_c24.integers(0, 2, size=length_c24)
        want_c24 = rates_c24(years_c24)
        got_c24 = rates_24(years_c24.copy())
        if got_c24 is None or np.shape(got_c24) != want_c24.shape or not np.allclose(got_c24, want_c24, rtol=1e-9,
                                                                                    atol=1e-12):
            wrong_c24.append(f"{length_c24} année(s) {years_c24[:9].tolist()}{'…' if length_c24 > 9 else ''} : attendu "
                             f"{np.round(want_c24[:4], 4).tolist()}…, reçu {shown(got_c24)[:80]}")
    report_24(not wrong_c24, "rates_24 : juste sur trois suites d'années que tu ne vois pas",
              "rates_24, " + " ; ".join(wrong_c24[:2]))
    curves_c24 = [rates_c24(YEARS_24)] + [rates_c24(np.random.default_rng(seed).integers(0, 2, size=20000))
                                          for seed in OTHER_SEEDS_24]
    mean_c24 = math.log(float(np.abs(np.linalg.eigvals((G_24 + M_24) / 2)).max()))
    bound_c24 = (math.log(float(np.linalg.svdvals(G_24)[0])) + math.log(float(np.linalg.svdvals(M_24)[0]))) / 2
    steps_c24 = np.arange(1, 20001)
    fig_24, ax_24 = plt.subplots(figsize=(8, 4.5))
    try:
        opened_24 = drawn(draw_rates_24, ax_24, curves_c24, mean_c24, bound_c24)
        report_24(not opened_24, "draw_rates_24 dessine dans l'axe reçu",
                  "draw_rates_24 doit dessiner dans l'axe reçu, sans ouvrir de figure")
        report_24(all(through(ax_24, steps_c24, curve) for curve in curves_c24),
                  "les six taux sont tracés contre n = 1, …, 20 000",
                  "il manque une des six courbes, tracée contre n = 1, …, 20 000 (une valeur par année)")
        report_24(ax_24.get_xscale() == "log", "l'abscisse est logarithmique",
                  "mets l'échelle des abscisses en logarithmique (ax.set_xscale(\"log\"))")
        report_24(horizontal(ax_24, mean_c24) and horizontal(ax_24, bound_c24), "les deux droites a) et d) sont tracées",
                  "il manque une droite horizontale : ln ρ(𝔼[A]), ou la majoration 𝔼[ln ‖A‖₂] (ax.axhline)")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_24)
        raise
    plt.show()
    verdict("M22.24", all(results_24), "tes taux sont justes, et dessinés avec les deux droites.", "")

**Après la figure**, dans ta copie : pourquoi ne pas utiliser `product_norms` sur les 20 000 années (regarde l'ordre de grandeur de $\lVert\mathbf{P}_n\rVert$) ? Pourquoi la population moyenne grandit-elle alors qu'une trajectoire typique s'éteint ? Pourquoi la majoration par les normes est-elle si loin du taux mesuré ?

### Ex M22.25 — Les spirales d'une récurrence 2 × 2 à valeurs propres complexes, selon leur module 🎨 ★ ⏱️ 15 min
**Objectif :** voir qu'une récurrence 2 × 2 à valeurs propres complexes tourne dans un repère oblique, et que le module des valeurs propres décide seul si elle rentre, tourne en rond ou sort.  
**Prérequis :** fiche §2 ; Ex M22.3 et M22.19 · **Parcours :** C, E

$\mathbf{A} = \begin{pmatrix} 0 & -1 \\ 1 & 0{,}6 \end{pmatrix}$ (`A_25`) a pour déterminant 1 et pour valeurs propres $e^{\pm i\theta}$, avec $\cos\theta = 0{,}3$ ; les valeurs propres de $r\mathbf{A}$ sont de module $r$. La cellule fournie donne les quatre valeurs $r = 0{,}6$, $0{,}93$, $1$ et $1{,}04$ (`RS_25`), $\mathbf{x}_0 = (1, 0)$ (`X0_25`) et $T = 60$ (`T_25`). Écris :
- `spiral_25(r)` : la trajectoire $\mathbf{x}_t = (r\mathbf{A})^t\mathbf{x}_0$ pour $t = 0, \ldots, 60$, un état par ligne (un tableau de forme $(61, 2)$), avec ta `linear_system_states` (une entrée nulle : `B = np.zeros((2, 1))` et `inputs = np.zeros((60, 1))`) ;
- `draw_spirals_25(fig, axes, trajectories)`, qui reçoit cinq axes côte à côte et les quatre trajectoires, dans l'ordre de `RS_25`, et dessine, sans ouvrir d'autre figure : dans chacun des quatre premiers axes, une trajectoire (les points reliés, dans l'ordre), avec la même échelle sur les deux axes (`ax.set_aspect("equal")`) et un titre qui donne $r$ et le module des valeurs propres ; dans le cinquième, $\lVert\mathbf{x}_t\rVert_2$ contre $t$ pour les quatre trajectoires, avec les courbes $r^t$ aux mêmes $t$, en échelle logarithmique, avec une légende.

La vérification compare `spiral_25` aux puissances de $r\mathbf{A}$, puis appelle `draw_spirals_25` et regarde le dessin, en s'arrêtant au premier ❌. Ce qu'il faut observer : la matrice tourne d'environ $\theta \approx 1{,}27$ rad (environ 73°) à chaque pas, un tour en un peu moins de 5 pas, dans un repère oblique ; le module des valeurs propres, $r$, décide seul : spirale qui rentre ($r < 1$), ellipse ($r = 1$), spirale qui sort ($r > 1$) ; et $\lVert\mathbf{x}_t\rVert$ suit $r^t$ en oscillant autour, parce que la trajectoire est une ellipse et non un cercle.

In [ ]:
A_25 = np.array([[0.0, -1.0], [1.0, 0.6]])          # det 1, eigenvalues e^(±iθ) with cos θ = 0.3
RS_25 = [0.6, 0.93, 1.0, 1.04]                       # the moduli r of the eigenvalues of r A_25
X0_25 = np.array([1.0, 0.0])
T_25 = 60

In [ ]:
def spiral_25(r):
    """The trajectory x_t = (r A_25)^t X0_25 for t = 0, ..., T_25, one state per row (an array (T_25 + 1, 2)), with
    your linear_system_states and zero inputs."""
    raise NotImplementedError("spiral_25 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_spirals_25(fig, axes, trajectories):
    """axes: 5 axes side by side; trajectories: the 4 trajectories, in the order of RS_25. In axes[i] (i < 4): the
    trajectory i (points joined, in order), equal scales on both axes, a title that gives r and the modulus of the
    eigenvalues; in axes[4]: ‖x_t‖₂ against t for the 4 trajectories and the curves r^t, the y scale logarithmic, a
    legend. No other figure."""
    raise NotImplementedError("draw_spirals_25 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M22.25"):
    results_25 = []
    report_25 = reporter(results_25)
    paths_c25 = [np.array([np.linalg.matrix_power(r * A_25, t) @ X0_25 for t in range(T_25 + 1)]) for r in RS_25]
    wrong_c25 = []
    for r_c25, want_c25 in zip(RS_25, paths_c25):
        name_c25 = f"{r_c25:g}".replace(".", ",")
        got_c25 = spiral_25(r_c25)
        if got_c25 is None or np.shape(got_c25) != want_c25.shape or not np.allclose(got_c25, want_c25, rtol=1e-9,
                                                                                    atol=1e-12):
            wrong_c25.append(f"r = {name_c25} : attendu un tableau {want_c25.shape} qui commence par "
                             f"{np.round(want_c25[:2], 4).tolist()}, reçu {shown(got_c25)[:90]}")
    report_25(not wrong_c25, "spiral_25 : les quatre trajectoires sont justes (comparées aux puissances de r A)",
              "spiral_25, " + " ; ".join(wrong_c25[:2]))
    fig_25, axes_25 = plt.subplots(1, 5, figsize=(22, 4.4))
    try:
        opened_25 = drawn(draw_spirals_25, fig_25, list(axes_25), paths_c25)
        report_25(not opened_25, "draw_spirals_25 dessine dans les axes reçus",
                  "draw_spirals_25 doit dessiner dans les cinq axes reçus, sans ouvrir de figure")
        for ax_c25, r_c25, path_c25 in zip(axes_25[:4], RS_25, paths_c25):
            name_c25 = f"{r_c25:g}".replace(".", ",")
            report_25(through(ax_c25, path_c25[:, 0], path_c25[:, 1]), f"r = {name_c25} : la trajectoire est tracée",
                      f"il manque, dans l'axe de r = {name_c25}, la trajectoire (ses {T_25 + 1} points, reliés, dans "
                      "l'ordre)")
            report_25(ax_c25.get_aspect() in ("equal", 1.0), "… avec la même échelle sur les deux axes",
                      f"l'axe de r = {name_c25} doit avoir la même échelle sur les deux axes (ax.set_aspect(\"equal\"))")
            numbers_c25 = [float(n.replace(",", ".")) for n in re.findall(r"\d+(?:[.,]\d+)?", ax_c25.get_title())]
            report_25(any(abs(n - r_c25) < 1e-9 for n in numbers_c25), "… et un titre qui donne r",
                      f"le titre de l'axe de r = {name_c25} doit donner r et le module des valeurs propres")
        steps_c25 = np.arange(T_25 + 1)
        report_25(axes_25[4].get_yscale() == "log", "le cinquième panneau est en échelle logarithmique",
                  "mets le cinquième panneau en échelle logarithmique (ax.set_yscale(\"log\"))")
        report_25(all(through(axes_25[4], steps_c25, np.linalg.norm(path, axis=1)) for path in paths_c25),
                  "les quatre normes ‖xₜ‖₂ sont tracées contre t",
                  "il manque, dans le cinquième panneau, une des quatre courbes ‖xₜ‖₂ contre t = 0, …, 60")
        report_25(all(through(axes_25[4], steps_c25, r ** steps_c25.astype(float)) for r in RS_25),
                  "… avec les quatre courbes rᵗ", "il manque, dans le cinquième panneau, une des quatre courbes rᵗ")
    except StopCheck:
        plt.show()
        raise
    except BaseException:
        plt.close(fig_25)
        raise
    plt.show()
    verdict("M22.25", all(results_25), "les quatre trajectoires et leurs normes sont dessinées.", "")

**Après la figure**, dans ta copie : combien de pas faut-il pour un tour ? Pourquoi la trajectoire de $r = 1$ est-elle une ellipse et pas un cercle ? Pourquoi $\lVert\mathbf{x}_t\rVert$ oscille-t-il autour de $r^t$, et avec quelle période ?

## Partie C · Déboguer, tester

Fiche §3, §4, §6 et §8. Trois erreurs classiques qui ne lèvent aucune erreur Python ; un test de propriété pour les normes induites. La vérification de M22.27 s'appuie sur la référence (`ref_rec`), pour ne pas dépendre de ton code.

In [ ]:
ref_rec = wbm.load_mymath("ref", alias="mymath_ref").recurrence   # the reference: the check of M22.27

### Ex M22.26 — Chasse à l'erreur : ρ(A) pris pour ‖A‖, un produit de jacobiennes dans le mauvais ordre, un seuil appliqué bloc par bloc 🐛 ★ ⏱️ 20 min
**Objectif :** reconnaître trois erreurs classiques qui ne lèvent aucune erreur Python et passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §3, §4, §6 et §8 ; Ex M22.8 et M22.10 · **Parcours :** C, E

Un collègue a écrit trois fonctions (cellule fournie) ; chacune a l'erreur que nomme le titre, aucune ne lève d'erreur Python, et chacune a passé la vérification qu'il a faite :
1. `max_stretch(A)` doit rendre le plus grand facteur par lequel $\mathbf{A}$ peut allonger un vecteur, en norme 2 ; elle renvoie `np.abs(np.linalg.eigvals(A)).max()`. Il l'a vérifiée sur $\begin{pmatrix} 2 & 1 \\ 1 & 2 \end{pmatrix}$, pour laquelle elle rend 3, et l'applique à $\begin{pmatrix} 2 & 3 \\ -1 & -2 \end{pmatrix}$ (`STRETCH_26`).
2. `trajectory_jacobian(jac, states)` doit rendre $\frac{\partial\mathbf{x}_T}{\partial\mathbf{x}_0} = \mathbf{J}_f(\mathbf{x}_{T-1})\cdots\mathbf{J}_f(\mathbf{x}_0)$ à partir de la fonction jacobienne `jac` et des états `states` $= [\mathbf{x}_0, \ldots, \mathbf{x}_{T-1}]$ ; elle part de $\mathbf{P} = \mathbf{I}$ et fait `P = P @ jac(x)` pour chaque état, dans l'ordre. Il l'a vérifiée sur l'application linéaire $\mathbf{x} \mapsto \begin{pmatrix} 0{,}6 & 0{,}2 \\ 0{,}1 & 0{,}3 \end{pmatrix}\mathbf{x}$, et l'applique au modèle proie-prédateur $f(x, y) = (1{,}3\,x - 0{,}1\,xy,\ 0{,}8\,y + 0{,}1\,xy)$ ($x$ les proies, $y$ les prédateurs, en milliers ; `prey_predator_26`, sa jacobienne `prey_predator_jacobian_26`), sur trois pas depuis $(4, 2)$ (les états $\mathbf{x}_0, \mathbf{x}_1, \mathbf{x}_2$ sont dans `STATES_26`).
3. `cap_speeds(V, c)` reçoit les vitesses d'une flotte de drones, une ligne par drone (un tableau de forme $(p, 3)$), et doit rendre un **nouveau** tableau dont la norme globale est au plus $c$, sans changer la direction de l'ensemble ; elle multiplie chaque ligne de norme supérieure à $c$ par $\frac{c}{\lVert\text{ligne}\rVert}$. Il l'a vérifiée sur un seul drone, $\mathbf{V} = \begin{pmatrix} 4 & -4 & 2 \end{pmatrix}$ et $c = 5$, et l'applique à la flotte $\mathbf{V} = \begin{pmatrix} -2 & 1 & 2 \\ 4 & -4 & 2 \\ 6 & 2 & -3 \\ 0 & 0 & 2 \end{pmatrix}$ (`FLEET_26`), avec $c = 5$ (`C_26`).

À faire : écris `max_stretch_fixed(A)`, `trajectory_jacobian_fixed(jac, states)` et `cap_speeds_fixed(V, c)` (qui doit rendre un tableau nul inchangé, sans division par 0, et ne jamais modifier le tableau reçu), puis réponds :
- a) `max_stretch_fixed` pour `STRETCH_26` (la valeur que donne le calcul ; 4 décimales dans ta copie) ;
- b) `trajectory_jacobian_fixed` pour le modèle proie-prédateur, sur ses trois pas (une matrice, la valeur que donne le calcul) ;
- c) la norme globale de la flotte rendue par la version fausse, `cap_speeds(FLEET_26, C_26)` ;
- d) la norme globale de la flotte rendue par ta version corrigée ;
- e) la première ligne rendue par ta version corrigée (une liste de 3 nombres).

Pour chacune des trois erreurs, dis aussi, dans ta copie, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie tes corrections sur des cas que tu ne vois pas : des matrices non symétriques (dont une nilpotente), des jacobiennes qui ne commutent pas, des flottes au-dessus et au-dessous du seuil, et une flotte immobile.

In [ ]:
STRETCH_26 = np.array([[2.0, 3.0], [-1.0, -2.0]])
FLEET_26 = np.array([[-2.0, 1.0, 2.0], [4.0, -4.0, 2.0], [6.0, 2.0, -3.0], [0.0, 0.0, 2.0]])   # one row per drone
C_26 = 5.0


def prey_predator_26(p):
    """f(x, y) = (1.3 x − 0.1 x y, 0.8 y + 0.1 x y): x the prey, y the predators, in thousands."""
    x, y = p
    return np.array([1.3 * x - 0.1 * x * y, 0.8 * y + 0.1 * x * y])


def prey_predator_jacobian_26(p):
    """Its jacobian at p, one row per output coordinate."""
    x, y = p
    return np.array([[1.3 - 0.1 * y, -0.1 * x], [0.1 * y, 0.8 + 0.1 * x]])


STATES_26 = [np.array([4.0, 2.0])]
for _ in range(2):
    STATES_26.append(prey_predator_26(STATES_26[-1]))   # x_0, x_1, x_2: three steps from (4, 2)


# The colleague's code
def max_stretch(A):
    """The largest factor by which A can stretch a vector, in the norm 2."""
    return float(np.abs(np.linalg.eigvals(A)).max())


def trajectory_jacobian(jac, states):
    """∂x_T/∂x_0 = J(x_(T−1)) ⋯ J(x_0), from the jacobian function jac and the states [x_0, ..., x_(T−1)]."""
    P = np.eye(len(states[0]))
    for x in states:
        P = P @ jac(x)
    return P


def cap_speeds(V, c):
    """A new array of speeds (one row per drone) whose global norm is at most c, the direction of the whole kept."""
    V = np.array(V, dtype=float)
    for i, row in enumerate(V):
        if np.linalg.norm(row) > c:
            V[i] = row * (c / np.linalg.norm(row))
    return V


# The colleague's checks: all pass
print("(1) max_stretch([[2, 1], [1, 2]]) =", max_stretch(np.array([[2.0, 1.0], [1.0, 2.0]])), "(attendu 3)")
LINEAR_26 = np.array([[0.6, 0.2], [0.1, 0.3]])
print("(2) x ↦ L x sur 3 pas :", bool(np.allclose(trajectory_jacobian(lambda x: LINEAR_26, [np.ones(2)] * 3),
                                                   np.linalg.matrix_power(LINEAR_26, 3))), "(attendu True)")
one_drone_26 = cap_speeds(np.array([[4.0, -4.0, 2.0]]), 5.0)
print("(3) un seul drone (4, −4, 2), c = 5 :", np.round(one_drone_26, 4).tolist(), "; norme",
      round(float(np.linalg.norm(one_drone_26)), 4), "(attendu 5, la même direction)")

In [ ]:
def max_stretch_fixed(A):
    """The largest factor by which A can stretch a vector, in the norm 2: a float."""
    raise NotImplementedError("max_stretch_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def trajectory_jacobian_fixed(jac, states):
    """∂x_T/∂x_0 = J(x_(T−1)) ⋯ J(x_0), from the jacobian function jac and the states [x_0, ..., x_(T−1)]."""
    raise NotImplementedError("trajectory_jacobian_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def cap_speeds_fixed(V, c):
    """A NEW array of speeds (one row per drone) whose global norm is at most c, the direction of the whole kept;
    a zero array is returned unchanged (a copy), without a division by 0. V is never modified."""
    raise NotImplementedError("cap_speeds_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_M22_26a = answer_M22_26b = answer_M22_26c = answer_M22_26d = answer_M22_26e = ...
with wbm.attempt("M22.26"):          # your three functions: ⏳ while they are not written
    answer_M22_26a = ...  # a) a number, as computed: max_stretch_fixed(STRETCH_26)
    answer_M22_26b = ...  # b) a 2 × 2 matrix, as computed: trajectory_jacobian_fixed for the prey–predator map, 3 steps
    answer_M22_26c = ...  # c) a number, as computed: the global norm of cap_speeds(FLEET_26, C_26), the wrong version
    answer_M22_26d = ...  # d) a number, as computed: the global norm of cap_speeds_fixed(FLEET_26, C_26)
    answer_M22_26e = ...  # e) a list of 3 numbers, as computed: the first row of cap_speeds_fixed(FLEET_26, C_26)

In [ ]:
for letter, answer in zip("abcde", [answer_M22_26a, answer_M22_26b, answer_M22_26c, answer_M22_26d,
                                       answer_M22_26e]):
    wbm.check(f"M22.26{letter}", answer, computed=True)

results_26, parts_26 = [], []


def report_26(ok, success, failure):
    results_26.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_26(part, name, function, *args):
    """function(*args), a RuntimeWarning (a division by 0) raised as an error; an error of the learner's code becomes
    one ❌ line that names the function (the check of this part stops there, the final verdict stays)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", RuntimeWarning)
            return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_26(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        parts_26.append(part)
        raise StopCheck from None


rng_c26 = np.random.default_rng(5716)
with checked("M22.26"):                                       # (1) max_stretch_fixed
    wrong_26 = []
    cases_c26 = [rng_c26.normal(size=(n, n)) for n in (2, 3, 4, 3)] + [np.array([[0.0, 2.5], [0.0, 0.0]])]
    for A_c26 in cases_c26:
        got_c26 = call_26(1, "max_stretch_fixed", max_stretch_fixed, A_c26.copy())
        want_c26 = float(np.linalg.svdvals(A_c26)[0])
        if got_c26 is None or np.ndim(got_c26) != 0 or not math.isclose(float(got_c26), want_c26, rel_tol=1e-9):
            wrong_26.append(f"une matrice {A_c26.shape[0]} × {A_c26.shape[1]} non symétrique : attendu "
                            f"{want_c26:.6f}, reçu {shown(got_c26, 6)}"[:220])
    report_26(not wrong_26, "(1) max_stretch_fixed : cinq matrices non symétriques, dont une nilpotente (ρ = 0)",
              "(1) max_stretch_fixed : " + " ; ".join(wrong_26[:2]))
    parts_26.append(1)

with checked("M22.26"):                                       # (2) trajectory_jacobian_fixed
    wrong_26 = []

    def twist_c26(x):
        """A jacobian that turns differently at each point: the factors do not commute."""
        return np.array([[1.0 + x[0], x[1] - 2.0], [0.5 * x[1], 1.0 - x[0] * x[1]]])

    for length_c26 in (1, 2, 5):
        states_c26 = [rng_c26.normal(size=2) for _ in range(length_c26)]
        want_c26 = np.eye(2)
        for x_c26 in states_c26:
            want_c26 = twist_c26(x_c26) @ want_c26                 # the last jacobian on the left
        got_c26 = call_26(2, "trajectory_jacobian_fixed", trajectory_jacobian_fixed, twist_c26,
                          [x.copy() for x in states_c26])
        if got_c26 is None or np.shape(got_c26) != (2, 2) or not np.allclose(got_c26, want_c26, rtol=1e-9, atol=1e-12):
            wrong_26.append(f"{length_c26} état(s), des jacobiennes qui ne commutent pas : attendu "
                            f"{np.round(want_c26, 4).tolist()}, reçu {shown(got_c26)}"[:240])
    report_26(not wrong_26, "(2) trajectory_jacobian_fixed : des trajectoires de 1, 2 et 5 états, des jacobiennes qui "
              "ne commutent pas", "(2) trajectory_jacobian_fixed : " + " ; ".join(wrong_26[:2]))
    parts_26.append(2)

with checked("M22.26"):                                       # (3) cap_speeds_fixed
    wrong_26 = []
    fleets_c26 = [(rng_c26.normal(size=(5, 3)) * 4, 3.0), (rng_c26.normal(size=(2, 3)) * 0.2, 4.0),
                  (np.zeros((3, 3)), 2.0), (rng_c26.normal(size=(1, 3)) * 5, 1.5)]
    for V_c26, c_c26 in fleets_c26:
        kept_c26 = V_c26.copy()
        size_c26 = float(np.sqrt(np.sum(V_c26 ** 2)))
        want_c26 = V_c26 * (min(1.0, c_c26 / size_c26) if size_c26 > 0 else 1.0)
        got_c26 = call_26(3, "cap_speeds_fixed", cap_speeds_fixed, V_c26, c_c26)
        what_c26 = (f"une flotte de {len(V_c26)} drone(s), de norme globale {size_c26:.3g}, c = {c_c26:g}"
                    if size_c26 > 0 else f"une flotte immobile, c = {c_c26:g}").replace(".", ",")
        if got_c26 is None or np.shape(got_c26) != V_c26.shape or not np.allclose(got_c26, want_c26, rtol=1e-9,
                                                                                  atol=1e-12):
            wrong_26.append(f"{what_c26} : attendu {np.round(want_c26, 3).tolist()}, reçu {shown(got_c26, 3)}"[:240])
        elif not np.array_equal(V_c26, kept_c26):
            wrong_26.append(f"{what_c26} : ta fonction modifie le tableau reçu ; rends un nouveau tableau")
        elif np.shares_memory(np.asarray(got_c26), V_c26):
            wrong_26.append(f"{what_c26} : ta fonction rend le tableau reçu lui-même ; rends une copie")
    report_26(not wrong_26, "(3) cap_speeds_fixed : des flottes au-dessus et au-dessous du seuil, et une flotte "
              "immobile (sans division par 0)", "(3) cap_speeds_fixed : " + " ; ".join(wrong_26[:2]))
    parts_26.append(3)

if len(set(parts_26)) == 3:
    verdict("M22.26", all(results_26), "les trois fonctions corrigées passent tous les cas.",
            "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chaque erreur, pourquoi la vérification du collègue ne la voyait-elle pas (une matrice symétrique, une application linéaire, un seul drone), et quelle vérification l'aurait vue ? Pour (1), de combien la version fausse se trompe-t-elle sur `STRETCH_26`, et quel vecteur est allongé le plus ? Pour (3), de combien de degrés la direction de la flotte tourne-t-elle avec la version fausse ?

### Ex M22.27 — Un test de propriété : ‖AB‖ ≤ ‖A‖ ‖B‖ et ρ(A) ≤ ‖A‖ sur des matrices tirées au hasard 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire un test de propriété pour une norme de matrices, voir qu'il laisse passer deux versions fausses plausibles, puis le compléter pour qu'il les attrape.  
**Prérequis :** fiche §4 ; Ex M22.20 · **Parcours :** C

Un **test de propriété** vérifie, sur beaucoup de cas tirés au hasard, une loi que tout résultat juste respecte, sans connaître la réponse de chaque cas (M17, M21). Écris `property_test_27(norm)`, qui reçoit une fonction de même signature que `operator_norm`, `norm(A, ord)`, et :
- tire au hasard, 300 fois, avec un générateur `np.random.default_rng(5727)` créé dans la fonction, trois tailles $m$, $k$, $n$ entre 1 et 5, une matrice $\mathbf{A}$ de forme $(m, k)$ et une matrice $\mathbf{B}$ de forme $(k, n)$ gaussiennes, puis une matrice carrée $\mathbf{S}$ de taille $k$ ;
- vérifie, pour `ord` = 1, 2 et `np.inf`, par des `assert` : $\lVert\mathbf{A}\mathbf{B}\rVert \le \lVert\mathbf{A}\rVert\,\lVert\mathbf{B}\rVert$ et $\rho(\mathbf{S}) \le \lVert\mathbf{S}\rVert$ (`np.linalg.eigvals`), avec une marge relative de $10^{-12}$ pour l'arrondi ; et un cas d'égalité, $\mathbf{B} = c\,\mathbf{I}$ (un $c$ tiré au hasard), qui doit donner $\lVert\mathbf{A}\mathbf{B}\rVert = \lvert c\rvert\,\lVert\mathbf{A}\rVert$ ;
- donne à chaque `assert` un message qui dit **d'abord** ce qui était attendu (il commence par « expected »), **puis** le cas fautif ($m$, $k$, $n$, `ord`) ;
- rend le nombre de cas vérifiés.

La cellule fournie donne deux versions **fausses** de `operator_norm`, construites sur la référence (`ref_rec`) : `norm_frobenius`, qui rend la norme de Frobenius pour `ord=2`, et `norm_swapped`, qui échange les lignes et les colonnes (`ord=1` rend la plus grande somme d'une ligne, `ord=np.inf` celle d'une colonne). Lance ton test sur ta `operator_norm`, puis sur les deux versions fausses : constate qu'elles **passent** ton test, et explique pourquoi dans la cellule de texte prévue.

Écris alors `property_test_complete_27(norm)`, qui lance `property_test_27(norm)`, puis ajoute deux propriétés qui attrapent les versions fausses, pour `ord` = 1, 2 et `np.inf` : $\lVert\mathbf{I}_n\rVert = 1$ pour $n = 2, \ldots, 5$, et $\lVert\mathbf{A}\mathbf{x}\rVert_p \le \lVert\mathbf{A}\rVert_p\,\lVert\mathbf{x}\rVert_p$ pour les vecteurs de la base canonique et pour des vecteurs de signes $\pm1$ (sur des matrices tirées au hasard). Ton test complété doit passer sur ta `operator_norm` et échouer sur chacune des deux versions fausses, par un `assert` dont le message dit d'abord ce qui était attendu, puis laquelle des propriétés a échoué. La vérification lance tes deux tests sur la référence, où ils doivent passer (elle compte les appels et les ordres), sur les deux versions fausses, puis sur **ta** librairie. Ce qu'il faut retenir : la sous-multiplicativité et $\rho \le \lVert\cdot\rVert$ valent pour toute norme sous-multiplicative, pas seulement pour les normes induites ; un test de propriété ne remplace pas un oracle, il faut choisir des propriétés qui **distinguent** la bonne fonction des fausses plausibles.

In [ ]:
def norm_frobenius(A, ord=2):
    """Wrong: the Frobenius norm for ord = 2 (built on the reference)."""
    if ord == 2:
        return float(np.sqrt(np.sum(np.asarray(A, dtype=float) ** 2)))
    return ref_rec.operator_norm(A, ord)


def norm_swapped(A, ord=2):
    """Wrong: the rows and the columns swapped, ord = 1 gives the largest row sum and ord = np.inf the largest
    column sum (built on the reference)."""
    if ord == 1:
        return ref_rec.operator_norm(A, np.inf)
    if ord == np.inf:
        return ref_rec.operator_norm(A, 1)
    return ref_rec.operator_norm(A, ord)

In [ ]:
def property_test_27(norm):
    """300 random cases drawn with rng = np.random.default_rng(5727), created inside: sizes m, k, n from 1 to 5, A (m, k)
    and B (k, n) Gaussian, then a square S of size k. For ord = 1, 2, np.inf, assert ‖A B‖ <= ‖A‖ ‖B‖ and
    ρ(S) <= ‖S‖ (a relative margin 1e-12), and the equality case B = c I: ‖A (c I)‖ = |c| ‖A‖. Each message says
    first what was expected ("expected ..."), then the case (m, k, n, ord). Return the number of cases checked."""
    raise NotImplementedError("property_test_27 n'est pas encore écrite (04_indices.md si tu bloques)")


def property_test_complete_27(norm):
    """property_test_27(norm), then, for ord = 1, 2, np.inf: ‖I_n‖ = 1 for n = 2, ..., 5, and ‖A x‖ <= ‖A‖ ‖x‖ for
    the vectors of the canonical basis and vectors of signs ±1 (random matrices). Each message says first what was
    expected, then which property failed and the case."""
    raise NotImplementedError("property_test_complete_27 n'est pas encore écrite (04_indices.md si tu bloques)")

In [ ]:
with checked("M22.27"):
    results_27 = []
    report_27 = reporter(results_27, stop=False)
    calls_27 = []

    def counted_27(norm):
        """norm, its calls recorded in calls_27 as (shape, ord)."""
        def wrapper(A, ord=2):
            calls_27.append((np.shape(A), ord))
            return norm(A, ord)
        return wrapper

    def run_27(test, norm, label):
        """test(norm): None when it passes, else the message of its AssertionError (other errors: ❌, then stop)."""
        try:
            test(norm)
        except AssertionError as exc:
            return str(exc) or "(un assert sans message)"
        except (NotImplementedError, MymathMissingError):
            raise
        except Exception as exc:  # noqa: BLE001 - another error than an assert
            print(f"❌ Ex M22.27 : {test.__name__}({label}) lève {type(exc).__name__} au lieu d'un assert : "
                  f"{str(exc)[:200]}")
            raise StopCheck from None
        return None

    def form_27(message):
        """The text of a message without its numbers: what tells one property from another."""
        return re.sub(r"[-+]?\d[\d.,eE+-]*", "#", message)

    try:
        cases_27 = property_test_27(counted_27(ref_rec.operator_norm))
    except AssertionError as exc:
        print(f"❌ Ex M22.27 : property_test_27 échoue sur la référence, qui est juste : {str(exc)[:300]}")
        raise StopCheck from None
    ords_27 = {o for _, o in calls_27}
    report_27(isinstance(cases_27, int) and cases_27 >= 300 and len(calls_27) >= 3 * 300,
              f"sur la référence, property_test_27 passe : {cases_27} cas, {len(calls_27)} appels de la norme",
              f"sur la référence, property_test_27 rend {cases_27!r} et fait {len(calls_27)} appels de la norme : "
              "300 cas tirés, chacun dans les trois ordres, et le nombre de cas rendu")
    report_27(ords_27 == {1, 2, np.inf} and any(s[0] != s[1] for s, _ in calls_27 if len(s) == 2),
              "les trois ordres sont testés, sur des matrices rectangulaires aussi",
              "ton test doit appeler la norme avec ord = 1, 2 et np.inf, et sur des matrices (m, k) et (k, n) "
              "rectangulaires")
    for wrong_27, label_27 in ((norm_frobenius, "norm_frobenius"), (norm_swapped, "norm_swapped")):
        message_27 = run_27(property_test_27, wrong_27, label_27)
        report_27(message_27 is None, f"{label_27} passe aussi property_test_27, comme annoncé",
                  f"property_test_27 attrape déjà {label_27} ; il ne doit vérifier que la sous-multiplicativité, "
                  f"ρ ≤ ‖·‖ et le cas B = c I, que {label_27} vérifie aussi : relis ton test ({message_27!r:.160})")
    message_ref_27 = run_27(property_test_complete_27, ref_rec.operator_norm, "référence")
    report_27(message_ref_27 is None, "property_test_complete_27 passe sur la référence",
              f"property_test_complete_27 échoue sur la référence, qui est juste : {message_ref_27!r:.200}")
    messages_27 = {}
    for wrong_27, label_27 in ((norm_frobenius, "norm_frobenius"), (norm_swapped, "norm_swapped")):
        message_27 = run_27(property_test_complete_27, wrong_27, label_27)
        messages_27[label_27] = message_27
        if message_27 is None:
            report_27(False, "", f"property_test_complete_27 n'attrape pas {label_27} (une AssertionError attrapée "
                      "dans ta fonction ne compte pas : laisse-la sortir)")
            continue
        report_27(True, f"property_test_complete_27 attrape {label_27} : {message_27[:160]}", "")
        report_27(message_27.strip().lower().startswith(("expected", "attendu")),
                  "… et son message dit d'abord ce qui était attendu",
                  f"le message doit commencer par ce qui était attendu (« expected … »), reçu : {message_27[:160]!r}")
    if all(messages_27.values()):
        report_27(form_27(messages_27["norm_frobenius"]) != form_27(messages_27["norm_swapped"]),
                  "les deux messages disent quelle propriété a échoué",
                  "les deux messages ont le même texte, aux nombres près : dis dans chacun quelle propriété a échoué")
    try:
        property_test_complete_27(mymath.recurrence.operator_norm)
        print("✅ ta librairie (operator_norm) passe ton test complété")
    except (NotImplementedError, MymathMissingError):
        print("⏳ ta librairie : operator_norm n'est pas encore écrite (M22.20)")
    except AssertionError as exc:
        print(f"❌ ta librairie échoue à ton test ({str(exc)[:200]}) : si ton test passe sur la référence, l'erreur "
              "est dans ton operator_norm (lance aussi ses tests, M22.20)")
    verdict("M22.27", all(results_27), "ton test passe sur la référence et attrape les deux versions fausses.",
            "relis les lignes ❌ ci-dessus.")

**Pourquoi les deux versions fausses passent-elles `property_test_27` ?** (ta réponse, en quelques phrases) …

**Après la vérification**, dans ta copie : quelle propriété attrape chacune des versions fausses, et pourquoi ? Donne une matrice $2 \times 2$ sur laquelle `norm_swapped` se trompe pour `ord=1`. Pourquoi les vecteurs de signes servent-ils pour $\lVert\cdot\rVert_\infty$, et ceux de la base canonique pour $\lVert\cdot\rVert_1$ (fiche §4, ∂ M22.12) ?

## Partie D · Défi

Fiche §3 et §4. Une matrice de rayon spectral 0,85 dont une puissance dépasse 100, avec des coefficients bornés.

### Ex M22.28 — Défi : une matrice 3 × 3 à coefficients dans [−3, 3], de rayon spectral 0,85, dont une puissance a une norme ‖·‖₂ supérieure à 100 🏆 ★★★ ⏱️ 25 min
**Objectif :** construire une matrice de rayon spectral < 1 dont une puissance grandit beaucoup, avec des coefficients bornés.  
**Prérequis :** fiche §3 et §4 ; Ex M22.5, M22.20 et M22.23 · **Parcours :** C

Construis une matrice $\mathbf{A}$ de forme $(3, 3)$, de coefficients tous dans $[-3, 3]$, de rayon spectral $\rho(\mathbf{A}) = 0{,}85$ (à $10^{-9}$ près), telle que $\lVert\mathbf{A}^t\rVert_2 > 100$ pour un pas $t \le 200$. Montre d'abord, dans la cellule de texte prévue, qu'il faut $t \ge 3$ : majore $\lVert\mathbf{A}\rVert_2$ par $\lVert\mathbf{A}\rVert_F$, puis $\lVert\mathbf{A}^2\rVert_2$.

**Critère de réussite :** la cellule de vérification contrôle la forme et les coefficients de ta matrice `challenge_28`, calcule $\rho(\mathbf{A})$ par `np.linalg.eigvals`, puis $\lVert\mathbf{A}^t\rVert_2$ pour $t = 1, \ldots, 200$ (par produits successifs et ta `operator_norm`), et réussit si $\lvert\rho(\mathbf{A}) - 0{,}85\rvert \le 10^{-9}$ et si l'une de ces normes dépasse 100. Puis relève, pour ta matrice :
- a) le premier pas $t$ où $\lVert\mathbf{A}^t\rVert_2 > 100$ (un entier) ;
- b) le plus grand $\lVert\mathbf{A}^t\rVert_2$ pour $t \le 200$, et le pas où il est atteint (une liste de 2 nombres : la norme, la valeur que donne le calcul, puis le pas).

La vérification compare a) et b) aux normes de ta matrice. Pour aller plus loin : peut-on réussir avec une matrice $2 \times 2$ de coefficients dans $[-3, 3]$ ?

In [ ]:
answer_M22_28a = answer_M22_28b = ...
challenge_28 = ...        # your 3 × 3 matrix (a list of 3 rows, or an array): coefficients in [-3, 3], ρ = 0.85
with wbm.attempt("M22.28"):          # your operator_norm: ⏳ while it is not written
    answer_M22_28a = ...  # a) an integer: the first step t where ‖A^t‖₂ > 100
    answer_M22_28b = ...  # b) a list [the largest ‖A^t‖₂ for t <= 200 (as computed), the step t where it is reached]

In [ ]:
with checked("M22.28"):
    if not filled(challenge_28):
        print("⏳ Ex M22.28 : la vérification attend ta matrice challenge_28.")
        raise StopCheck
    try:
        A_c28 = np.array(challenge_28, dtype=float)
    except (TypeError, ValueError):
        verdict("M22.28", False, "", "challenge_28 doit être une matrice 3 × 3 de nombres (une liste de 3 lignes).")
        raise StopCheck from None
    reasons_28 = []
    if A_c28.shape != (3, 3) or not np.all(np.isfinite(A_c28)):
        verdict("M22.28", False, "", f"challenge_28 doit être une matrice 3 × 3 de nombres finis, reçu la forme "
                f"{A_c28.shape}.")
        raise StopCheck
    if not np.all(np.abs(A_c28) <= 3):
        reasons_28.append(f"un coefficient sort de [−3, 3] (le plus grand en valeur absolue vaut "
                          f"{np.abs(A_c28).max():.4g})".replace(".", ","))
    rho_c28 = float(np.abs(np.linalg.eigvals(A_c28)).max())
    if abs(rho_c28 - 0.85) > 1e-9:
        reasons_28.append(f"ρ(A) = {rho_c28:.10g}".replace(".", ",") + ", pas 0,85 à 10⁻⁹ près")
    if reasons_28:
        verdict("M22.28", False, "", "critère pas encore atteint : " + " ; ".join(reasons_28) + ".")
        raise StopCheck
    norms_c28, P_c28 = [], np.eye(3)
    for t_c28 in range(1, 201):
        P_c28 = A_c28 @ P_c28
        norms_c28.append(mymath.recurrence.operator_norm(P_c28, 2))     # your operator_norm, as the statement says
    oracle_c28 = [float(np.linalg.svdvals(np.linalg.matrix_power(A_c28, t))[0]) for t in range(1, 201)]
    if not np.allclose(norms_c28, oracle_c28, rtol=1e-9, atol=1e-12):
        verdict("M22.28", False, "", "ton operator_norm ne rend pas s_max sur les puissances de ta matrice : corrige-la "
                "d'abord (lance ses tests, M22.20).")
        raise StopCheck
    peak_c28 = int(np.argmax(norms_c28)) + 1
    verdict("M22.28", max(norms_c28) > 100,
            f"critère atteint : ρ(A) = 0,85 et ‖Aᵗ‖₂ monte jusqu'à {fr(max(norms_c28), 2)} (t = {peak_c28}).",
            f"critère pas encore atteint : ρ(A) = 0,85, mais ‖Aᵗ‖₂ ne dépasse pas {fr(max(norms_c28), 2)} pour t ≤ 200.")
    if max(norms_c28) > 100 and filled(answer_M22_28a):
        first_c28 = 1 + next(i for i, n in enumerate(norms_c28) if n > 100)
        verdict("M22.28", answer_M22_28a == first_c28, "a) le premier pas où la norme dépasse 100 est juste.",
                "a) ce n'est pas le premier pas t où ‖Aᵗ‖₂ dépasse 100 pour ta matrice (les pas commencent à t = 1).")
    if max(norms_c28) > 100 and filled(answer_M22_28b):
        ok_b28 = (np.shape(answer_M22_28b) == (2,) and math.isclose(float(answer_M22_28b[0]), max(norms_c28),
                                                                     rel_tol=1e-9)
                  and int(answer_M22_28b[1]) == peak_c28)
        verdict("M22.28", ok_b28, "b) le maximum et son pas sont justes.",
                "b) attendu [le plus grand ‖Aᵗ‖₂ pour t ≤ 200, le pas t où il est atteint], pour ta matrice.")

**Pourquoi faut-il $t \ge 3$ ?** (ta majoration de $\lVert\mathbf{A}\rVert_2$, puis de $\lVert\mathbf{A}^2\rVert_2$) …

**Dans ta copie** : écris $\mathbf{A}^t$ pour ta matrice (la formule du binôme, fiche §3, si elle est triangulaire) ; quel terme fait grandir la norme, et jusqu'à quel $t$ ? Que se passe-t-il si tu changes le signe d'un coefficient au-dessus de la diagonale ?

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu résoudre une récurrence linéaire avec entrée, et lire sa sortie comme une convolution par sa réponse impulsionnelle ?
2. Sais-tu calculer les puissances d'une matrice, ses valeurs propres complexes sous forme exponentielle, et reconnaître une rotation et une homothétie ?
3. Sais-tu décider si $\mathbf{A}^t$ tend vers 0 par le rayon spectral, et repérer une croissance transitoire ?
4. Sais-tu calculer les normes induites $\lVert\cdot\rVert_1$, $\lVert\cdot\rVert_\infty$ et $\lVert\cdot\rVert_2$, et majorer la norme d'un produit ?
5. Sais-tu suivre un produit de facteurs dans [0, 1] ou de jacobiennes, écrire une sensibilité comme une somme sur le temps, et ramener un vecteur fait de blocs sous un seuil ?

**La suite** : le chapitre 22 de workbookIA, qui fait passer des séquences dans une même règle de mise à jour d'un état, regarde comment l'information et les dérivées se propagent sur de longues séquences, et limite la norme de ses mises à jour ; puis l'examen blanc CP5, qui clôt la partie V.